In [ ]:
import cv2
import torch
import torch.nn as nn
import torchvision
import torchvision.models as models
import torchvision.models.detection as detection
import torch.optim as optim
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
# from torch.utils.data import Dataset
from energy_meter_dataset import EnergyMeterDataset


In [ ]:
class Preprocessor:
    def __init__(self, image_size):
        self.image_size = image_size
    def __call__(self, image):
        image = cv2.resize(image, self.image_size)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
        image = cv2.equalizeHist(image)
        image = torch.from_numpy(image).unsqueeze(0).float()
        return image


In [ ]:
class ResNetTrainer:
    def __init__(self, num_classes):
        self.model = models.resnet50(pretrained=True)

        for param in self.model.parameters():
            param.requires_grad = False

        self.model.fc = nn.linear(self.model.fc.in_features, num_classes)

        self.criterion = nn.CrossEntropyLoss()
        self.optimizer = optim.SGD(self.model.fc.parameters(), lr=0.001, momentum=0.9)

    def train(self, dataloader):
        for epoch in range(10):
            for images, labels in dataloader:
                preprocessor = Preprocessor((224, 224))
                images = torch.stack([preprocessor(image) for image in images])
                outputs = self.model(images)
                loss = self.criterion(outputs, labels)

                self.optimzer.zero_grad()
                loss.backward()
                self.optimzer.step()

                

In [ ]:
class FasterRCNN:
    def __init__(self, num_classes):
        self.model = detection.fasterrcnn_resnet50_fpn(pretrained=True)

        in_features = self.model.roi_heads.box_predictor.cls_score.in_features
        self.model.roi_heads.box_predictor = detection.FasterRCNNPredictor(in_features, num_classes)

        self.criterion = detection.fasterrcnn_loss
        self.optimizer = optim.SGD(self.model.parameters(), lr=0.001, momentum=0.9)

    def train(self, dataloader):
        for epoch in range(10):
            for images, targets in dataloader:
                preprocessor = Preprocessor((224, 224))
                images = torch.stack([preprocessor(image) for image in images])

                targets = [{k : v.to(device) for k, v in t.items()} for t in targets]

                losses = self.model(images, targets)
                loss = sum(loss for loss in losses.values())

                self.optimize.zero_grad()
                loss.backward()
                self.optimizer.step()


In [ ]:
class DialDetector:
    def __init__(self, num_classes, train_dataset_dir):
        self.train_dataset_dir = train_dataset_dir
        self.num_classes = num_classes
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    def train(self):
        train_dataset = EnergyMeterDataset(self.train_dataset_dir)
        dataloader = DataLoader(train_dataset, batch_size=4, shuffle=True)

        resnet_trainer = ResNetTrainer(self.num_classes)
        resnet_trainer.train(dataloader)

        detector = FasterRCNN(self.num_classes)
        detector.model.to(self.device)
        detector.train(dataloader)

    def detect(self, image_path):
        image = cv2.imread(image_path)

        preprocessor = Preprocessor((224, 224))
        image = preprocessor(image)
        image = image.to(self.device)

        with torch.no_grad():
            detections = detector.model([image])
            image = image.squeeze(0).cpu().numpy()
            for box in detections[0]['boxes']:
                x1, y1, x2, y2 = box.cpu().numpy().astype(int)
                cv2.rectangle(image, (x1, y1), (x2, y2), (0, 255, 0), 2)

                cv2.imshow('image', image)
                cv2.waitKey(0)

In [ ]:
# class DialMeterDataset(Dataset):
#     def __init__(self, image_paths, annotations):
#         self.image_paths = image_paths
#         self.annotations = annotations
#         self.transform = transforms.Compose([
#             transforms.Resize((800, 800)),
#             transforms.ToTensor(),
#             transforms.Normalize((0,485, 0.456, 0.406), (0.229, 0.224, 0.225))
#         ])

#         def __getitem__(self, index):
#             image_path = self.image_paths[index]
#             image = Image.open(image_)